# Filtering and Fixing

In [ ]:
!pip install vllm==0.19.1

In [ ]:
! pip install -q json-repair

In [ ]:
!pip install -q  qwen-vl-utils python-docx

In [ ]:

# --- 2. Define the Regex Pattern & Tiers ---
tier_1_auto_keep = [
    'Shelling', 'Large explosive drone(Shahed)',
    'Rocket', 'Drones, robots', 'Cruise Missile', 'Explosion, blasts','Destruction','Map','Fires',

]

# tier_2_conditional = [
#     'Speech, statement', 'Twitter', 'Press', 'Video', 'Phone', 'Map','Rifle Gun, armed men',
#     'Fires', 'Destruction', 'Dead', 'Injures/medicine', 'Rescue operation',
#     'Air Alert', 'Airplanes, jets', 'Ship, Warship', 'Submarine', 'Helicopters'
# ]


# LLM

In [ ]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 66.0 MB/s eta 0:00:00


In [ ]:
!pip install -U -q transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 177.2 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import snapshot_download
import os
# ==============================================================================
# import os
# CRITICAL FIX: Forces safe multiprocessing to prevent the CUDA deadlock
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "INFO"

qwen_id = "Qwen/Qwen3.5-27B"


In [ ]:
from huggingface_hub import snapshot_download
import os
# ==============================================================================
# import os
# CRITICAL FIX: Forces safe multiprocessing to prevent the CUDA deadlock
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "INFO"
import json
import pandas as pd
import json_repair
import torch
import gc
import re
from transformers import AutoTokenizer, pipeline
from vllm import LLM, SamplingParams
qwen_id = "Qwen/Qwen3.5-27B"

print(f"\n--- STEP 0: Downloading {qwen_id} to Disk (CPU) ---")
# This will show a clear, visible progress bar for the 54GB download
snapshot_download(
    repo_id=qwen_id,

)
print("Download complete! Model is now cached on disk.")

print(f"\n--- STEP 1: Loading {qwen_id} into GPU ---")
qwen_llm = LLM(
    model=qwen_id,
    dtype="bfloat16",
    tensor_parallel_size=1,
    max_model_len=4096,
    gpu_memory_utilization=0.90,
    enforce_eager=True,
    disable_log_stats=False,
    quantization="bitsandbytes",
    load_format="bitsandbytes"
)


--- STEP 0: Downloading Qwen/Qwen3.5-27B to Disk (CPU) ---


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

Download complete! Model is now cached on disk.

--- STEP 1: Loading Qwen/Qwen3.5-27B into GPU ---
INFO 07-30 01:46:09 [utils.py:233] non-default args: {'load_format': 'bitsandbytes', 'dtype': 'bfloat16', 'max_model_len': 4096, 'quantization': 'bitsandbytes', 'enforce_eager': True, 'model': 'Qwen/Qwen3.5-27B'}
WARNING 07-30 01:46:09 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 07-30 01:46:22 [model.py:549] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 07-30 01:46:22 [model.py:1678] Using max model len 4096
INFO 07-30 01:46:22 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.


[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


INFO 07-30 01:46:23 [config.py:281] Setting attention block size to 784 tokens to ensure that attention page size is >= mamba page size.
INFO 07-30 01:46:23 [config.py:312] Padding mamba page size by 0.13% to ensure that mamba page size and attention page size are exactly equal.
INFO 07-30 01:46:23 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 07-30 01:46:23 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 07-30 01:46:23 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 07-30 01:46:23 [vllm.py:1025] Cudagraph is disabled under eager mode
INFO 07-30 01:46:23 [compilation.py:292] Enabled custom fusions: norm_quant, act_quant


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


In [ ]:
model_dir = "/content/drive/MyDrive/Iran Israel War/Qwen3.5-27B"

In [ ]:
# from huggingface_hub import snapshot_download
# import os
# # ==============================================================================
# # import os
# # CRITICAL FIX: Forces safe multiprocessing to prevent the CUDA deadlock
# os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# os.environ["VLLM_USE_V1"] = "0"
# os.environ["VLLM_LOGGING_LEVEL"] = "INFO"
# import json
# import pandas as pd
# import json_repair
# import torch
# import gc
# import re
# from transformers import AutoTokenizer, pipeline
# from vllm import LLM, SamplingParams
# qwen_id = "Qwen/Qwen3.5-27B"


# print(f"\n--- STEP 1: Loading {qwen_id} into GPU ---")
# qwen_llm = LLM(
#     model=model_dir,
#     dtype="bfloat16",
#     tensor_parallel_size=1,
#     max_model_len=4096,
#     gpu_memory_utilization=0.90,
#     enforce_eager=True,
#     disable_log_stats=False,
#     quantization="bitsandbytes",
#     load_format="bitsandbytes"
# )

In [ ]:
import os
# ==============================================================================
# import os
# CRITICAL FIX: Forces safe multiprocessing to prevent the CUDA deadlock
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "INFO"
import json
import pandas as pd
import json_repair
import torch
import gc
import re
from transformers import AutoTokenizer, pipeline
from vllm import LLM, SamplingParams

# ==============================================================================
# PREPARATION: Setup and Load Data
# ==============================================================================
base_folder = '/content/drive/MyDrive/Iran Israel War'
os.makedirs(base_folder, exist_ok=True)

# Load the raw dataset
csv_path = os.path.join(base_folder, 'liveuamap_merged_final.csv')
df = pd.read_csv(csv_path, on_bad_lines='skip')
tier_1_auto_keep = [
    'Shelling', 'Large explosive drone(Shahed)', 'Anti-air, SAM',
    'Rocket', 'Drones, robots', 'Cruise Missile', 'Explosion, blasts'
]

# Filter for relevant kinetic events
df = df[df['event_category'].isin(tier_1_auto_keep)].copy().head(10)

print(f"Loaded DataFrame with {len(df)} rows.")

Loaded DataFrame with 10 rows.


In [ ]:
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_id)

In [ ]:
# ==============================================================================
# STEP 1: QWEN REASONING & EXPANSION
# ==============================================================================
import re # Make sure re is imported
import pandas as pd
import json
import gc
import torch
import os
# Assuming qwen_llm, qwen_tokenizer, json_repair, SamplingParams, and df are already defined/loaded

generator_prompt = """You are an elite Intelligence Analyst.
Read the following military alert and write a highly structured intelligence brief.

*** MANDATORY ANALYTICAL SCRATCHPAD (<think> block) ***
Before writing your final brief, you MUST open a `<think>` block to analyze:

1. TARGET & SCOPE: Identify the specific proper noun or generic facility.
   - SCOPE CHECK: Is this a single distinct target, or does the text explicitly mention "several sites", "multiple areas", or a broad region? (Set multiple_sites_flag to "True" or "False").

2. GROUND IMPACT & STRUCTURAL DAMAGE:
   - STEP A (IMPACT STATUS): Classify into ONE of four strict statuses:
     * "Hit Ground": Detonated on a ground target.
     * "Debris": Intercepted, but falling debris impacted ground/caused fire.
     * "Intercepted Safe": Shot down mid-air with NO reported damage.
     * "Hit Ground (Assumed)": Fired towards an area; assume it hit.
   - STEP B (STRUCTURAL IMPACT): Based on the text and your deduction, what is the level of destruction? Pick ONE:
     * "Total Annihilation": Entire buildings leveled, massive craters, catastrophic damage.
     * "Severe Structural": Partial building collapse, heavy fire, severe damage.
     * "Targeted / Room-Level": Specific room/floor destroyed, building remains standing.
     * "Minor / Surface": Shattered glass, open field craters, shrapnel marks, minor fires.
     * "Unknown": No damage details provided.

3. WEAPON CLASSIFICATION:
   - STEP C (WEAPON IDENTIFICATION & QUANTITIES): Identify ALL weapon types mentioned and deduce their quantities. If a specific number isn't provided (e.g., "a swarm", "several", "a barrage"), estimate a logical integer based on the context. If singular, use 1.
   - Classify EACH identified weapon into one of these strict categories:
     * "Small Kamikaze Drone" (e.g., quadcopters, small FPVs)
     * "Heavy Loitering Munition" (e.g., Shahed-136)
     * "Unguided Rocket" (e.g., Grad, Qassam, Hezbollah rockets)
     * "Heavy Artillery / Mortar" (e.g., 155mm shells)
     * "Air-to-Surface / Smart Bomb" (e.g., Fighter jet strikes, JDAMs)
     * "Cruise Missile"
     * "Ballistic Missile"
     * "Unknown Generic"

*** URGENT DIRECTIVE ***
Write your brief inside `<intelligence_brief>` tags. Separate distinct targets with "EVENT [X]" headers.

CRITICAL ARRAY FORMATTING:
EVENT 1:
- specific_name: [Strict Proper Noun or Null]
- general_facility_name: [Strict Generic Facility or Null]
- facility_type: [Military / Civilian / Dual-Use / Unknown]
- multiple_sites_flag: [True / False]
- weapons_breakdown: [List EACH weapon class and its quantity, e.g., "6 Ballistic Missile, 10 Heavy Loitering Munition"]
- ground_impact_status: [Insert EXACT string from Step A]
- structural_impact: [Insert EXACT string from Step B]
"""

texts_to_process = df['event_full_title'].tolist()
indices_to_process = df.index.tolist()

qwen_prompts = []
for text in texts_to_process:
    if not isinstance(text, str) or text.strip() == "":
        qwen_prompts.append("")
    else:
        messages = [
            {"role": "system", "content": generator_prompt},
            {"role": "user", "content": f"Alert Text: {text}\n\nWrite the expanded brief."}
        ]
        prompt = qwen_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        qwen_prompts.append(prompt)

qwen_sampling = SamplingParams(temperature=0.4, top_p=0.9, max_tokens=1500)

print(f"Generating Intelligence Briefs for {len(qwen_prompts)} rows...")
qwen_outputs = qwen_llm.generate(qwen_prompts, qwen_sampling)

expanded_briefs = []
thinking_logs = [] # --- NEW: List to store extracted thinking ---

for idx, output in enumerate(qwen_outputs):
    current_index = indices_to_process[idx]

    if not qwen_prompts[idx]:
        expanded_briefs.append("")
        thinking_logs.append({'source_index': current_index, 'qwen_thinking': ""})
    else:
        raw_output_text = output.outputs[0].text.strip()
        combined_text = f"ORIGINAL TITLE: {texts_to_process[idx]}\nEXPANDED BRIEF: {raw_output_text}"
        expanded_briefs.append(combined_text)

        # --- NEW: Extract the text inside <think>...</think> ---
        think_match = re.search(r'<think>(.*?)</think>', raw_output_text, flags=re.DOTALL | re.IGNORECASE)
        extracted_thought = think_match.group(1).strip() if think_match else ""

        thinking_logs.append({
            'source_index': current_index,
            'qwen_thinking': extracted_thought,
            'expanded_brief': raw_output_text
        })

# --- NEW: Convert thinking logs to DataFrame for later merging ---
df_thinking = pd.DataFrame(thinking_logs)

# ==============================================================================
# STEP 1.5: VRAM FLUSH
# ==============================================================================
print("\n--- FLUSHING VRAM ---")
del qwen_llm
del qwen_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("VRAM cleared. Ready for Step 2.\n")

# ==============================================================================
# STEP 2: THE EXTRACTOR (NuExtract 1.5)
# ==============================================================================

nu_id = "numind/NuExtract-1.5"
print(f"--- STEP 2: Loading {nu_id} ---")

nu_llm = LLM(
    model=nu_id,
    dtype="bfloat16",
    tensor_parallel_size=1,
    max_model_len=8192,
    gpu_memory_utilization=0.90,
    enforce_eager=False,
    trust_remote_code=True
)

nu_template_dict = {
  "extracted_events": [
    {
      "target": {
        "specific_name": "",
        "general_facility_name": "",
        "facility_type": "",
        "multiple_sites_flag": ""
      },
     "munitions_and_damage": {
        "weapons_used": [
          {
            "weapon_class": "", # E.g., "Ballistic Missile"
            "estimated_quantity": 0 # E.g., 12 (use 1 if not specified)
          }
        ],
        "ground_impact_status": "",
        "structural_impact": ""
      }
    }
  ]
}

json_template = json.dumps(nu_template_dict, indent=2)

nu_prompts = []
for brief in expanded_briefs:
    if not brief:
        nu_prompts.append("")
    else:
        nu_prompt = f"<|input|>\n### Template:\n{json_template}\n### Text:\n{brief}\n\n<|predict|>"
        nu_prompts.append(nu_prompt)

nu_sampling = SamplingParams(temperature=0.0, max_tokens=4000)

print("Extracting strict JSON from expanded briefs...")
nu_outputs = nu_llm.generate(nu_prompts, nu_sampling)

# ==============================================================================
# STEP 3: JSON REPAIR & PYTHON PROGRAMMATIC INJECTION
# ==============================================================================
print("\n--- STEP 3: PARSING FINAL JSON ---")
all_results = []

for idx, output in enumerate(nu_outputs):
    current_index = indices_to_process[idx]

    if not nu_prompts[idx]:
        continue

    response_text = output.outputs[0].text.strip()
    start_idx = response_text.find('{')
    end_idx = response_text.rfind('}')

    if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
        clean_json = response_text[start_idx:end_idx+1]
        try:
            parsed_json = json_repair.loads(clean_json)
            if isinstance(parsed_json, str):
                parsed_json = json_repair.loads(parsed_json)

            if isinstance(parsed_json, dict):
                parsed_json["status"] = "SUCCESS"
                parsed_json["source_index"] = current_index
                all_results.append(parsed_json)
        except Exception:
            pass

# ==============================================================================
# STEP 4: PANDAS FORMATTING & DETERMINISTIC PHYSICS ENFORCER
# ==============================================================================
print("\n--- STEP 4: ENFORCING SINGLE IMPACT RADIUS & MERGING ---")

df_results = pd.DataFrame(all_results)
df_exploded = df_results.explode('extracted_events').dropna(subset=['extracted_events']).reset_index(drop=True)
df_exploded = df_exploded[df_exploded['extracted_events'].apply(lambda x: isinstance(x, dict) and len(x) > 0)]

df_json_expanded = pd.json_normalize(df_exploded['extracted_events']).reset_index(drop=True)
meta_df = df_exploded[['source_index']].reset_index(drop=True)
df_expanded = pd.concat([meta_df, df_json_expanded], axis=1)

# Strip prefixes
new_columns = {col: col.split('.')[-1] for col in df_expanded.columns if '.' in col}
df_expanded = df_expanded.rename(columns=new_columns)

# --- THE PHYSICS ENFORCER (SINGLE RADIUS) ---
def calculate_salvo_impact(row):
    weapons_list = row.get('weapons_used', [])
    impact = str(row.get('structural_impact', '')).lower()

    # Base blast radii dictionary
    radii_chart = {
        'small kamikaze': 5,
        'loitering': 15,
        'shahed': 15,
        'rocket': 12,
        'artillery': 20,
        'air-to-surface': 40,
        'cruise': 35,
        'ballistic': 60
    }

    max_single_radius = 0
    total_munitions = 0

    # 1. Loop through all weapons in the salvo
    if isinstance(weapons_list, list):
        for w in weapons_list:
            w_class = str(w.get('weapon_class', '')).lower()
            qty = int(w.get('estimated_quantity', 1))
            total_munitions += qty

            # Find the base radius for this specific weapon
            for key, val in radii_chart.items():
                if key in w_class:
                    if val > max_single_radius:
                        max_single_radius = val # Keep the heaviest weapon's radius
                    break

    # Default fallback if parsing fails
    if max_single_radius == 0: max_single_radius = 18
    if total_munitions == 0: total_munitions = 1

    # 2. Apply structural multipliers to the max radius
    multiplier = 1.0
    if 'total annihilation' in impact: multiplier += 0.5
    elif 'targeted' in impact: multiplier -= 0.5

    final_max_crater = int(max_single_radius * max(0.2, multiplier))

    # 3. Calculate "Total Facility Footprint" (spreading the damage out)
    # If 12 missiles hit, the facility damage spans much wider than 1 crater
    total_facility_footprint = final_max_crater + (total_munitions * 10)

    return final_max_crater, total_facility_footprint

# Apply to DataFrame
df_expanded[['max_crater_radius_m', 'total_damage_footprint_m']] = df_expanded.apply(
    lambda row: pd.Series(calculate_salvo_impact(row)), axis=1
)

# Merge JSON results back to the original dataframe
df_final = df.merge(df_expanded, left_index=True, right_on='source_index', how='right')
df_final = df_final.loc[:, ~df_final.columns.duplicated()]

# --- NEW: Merge the Qwen thinking logs into the final dataframe ---
df_final = df_final.merge(df_thinking, on='source_index', how='left')

print(f"Pipeline complete! Successfully calculated impact radii on {len(df_final)} total rows.")

export_filename = os.path.join('', "final_extracted_events.csv")
df_final.to_csv(export_filename, index=False, encoding="utf-8")
print(f"[SUCCESS] Cleaned data exported to: {export_filename}")

Generating Intelligence Briefs for 10 rows...


Rendering prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

INFO 07-30 01:49:19 [loggers.py:259] Engine 000: Avg prompt throughput: 110.1 tokens/s, Avg generation throughput: 0.2 tokens/s, Running: 10 reqs, Waiting: 0 reqs, GPU KV cache usage: 3.2%, Prefix cache hit rate: 0.0%


In [ ]:
from google.colab import runtime

# This command disconnects the runtime and deletes the virtual machine
runtime.unassign()